# Reader Node — Unit Tests
Step-by-step testing of the Reader Node functions.
Run each cell independently to inspect intermediate outputs and logs.

In [2]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

import os

from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE, OPENAPI_SPEC_DIR
from tools.document_tools import load_markdown, load_openapi_reference
from tools.web_tools import fetch_openapi_reference
from utils.parsers import parse_sections
from nodes.reader import reader_node

logger = get_logger(__name__)
logger.info("Imports OK.")

2026-03-29 22:29:06 [INFO] __main__: Imports OK.


In [3]:
# Step 2 — Test load_markdown()

# Happy path: load a real 3GPP file
text = load_markdown(TSPEC_DATA_TEST_FILE)
assert isinstance(text, str) and len(text) > 0, "Expected non-empty string"
logger.info(f"load_markdown OK — {len(text)} chars loaded from {TSPEC_DATA_TEST_FILE}")

2026-03-29 22:29:06 [INFO] __main__: load_markdown OK — 1008071 chars loaded from /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../../../Dataset/TSpec-LLM/3GPP-clean/Rel-18/28_series/28532-i00.md


In [4]:
# Step 3 — Test parse_sections() with the real 3GPP file
# Loads the file directly so this cell can run independently of Step 2.

raw_text = load_markdown(TSPEC_DATA_TEST_FILE)
sections = parse_sections(raw_text)

assert isinstance(sections, list), "Expected list"
assert len(sections) > 0, "Expected at least one relevant section"
assert all(isinstance(s, dict) for s in sections), "Expected list of dicts"
assert all("section_id" in s and "title" in s and "content" in s for s in sections)

logger.info(f"parse_sections OK — {len(sections)} relevant section(s) kept.")
for s in sections[:5]:  # preview first 5 sections
    logger.info(f"  [{s['section_id']}] {s['title']}")

2026-03-29 22:29:06 [INFO] __main__: parse_sections OK — 238 relevant section(s) kept.
2026-03-29 22:29:06 [INFO] __main__:   [1] 11.1.0 Introduction
2026-03-29 22:29:06 [INFO] __main__:   [4] 11.1.1.1.1 Description
2026-03-29 22:29:06 [INFO] __main__:   [5] 11.1.1.1.2 Input parameters
2026-03-29 22:29:06 [INFO] __main__:   [6] 11.1.1.1.3 Output parameters
2026-03-29 22:29:06 [INFO] __main__:   [8] 11.1.1.2 getMOIAttributes operation


In [5]:
# Step 4 — Fetch OpenAPI spec (run only if data/references/openapi_spec/ is empty)
# Calls fetch_openapi_reference() from tools. This is the same function main.py
# calls at pipeline startup to ensure the spec is available before the graph runs.

spec_files = [f for f in os.listdir(OPENAPI_SPEC_DIR) if f.endswith(".md")] if os.path.isdir(OPENAPI_SPEC_DIR) else []

if spec_files:
    logger.info(f"OpenAPI spec already present ({len(spec_files)} file(s)). Skipping fetch.")
else:
    logger.warning("OpenAPI spec not found — fetching...")
    fetch_openapi_reference()
    logger.info("Fetch complete.")

2026-03-29 22:29:06 [INFO] __main__: OpenAPI spec already present (1 file(s)). Skipping fetch.


In [6]:
# Step 4 — Test load_openapi_reference()
# Guard: returns "" with a warning if data/references/openapi_spec/ is empty.
# After running fetch_openapi_reference() from tools/web_tools.py, returns the full spec content.

context = load_openapi_reference()

if not context:
    logger.warning(
        "openapi_spec_context is empty. "
        "Run fetch_openapi_reference() from tools/web_tools.py and re-run this cell."
    )
else:
    assert isinstance(context, str) and len(context) > 0
    logger.info(f"load_openapi_reference OK — {len(context)} chars loaded.")
    logger.info(f"Preview: {context[:300]}")

2026-03-29 22:29:06 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-29 22:29:06 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-29 22:29:06 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_spec'.
2026-03-29 22:29:06 [INFO] __main__: load_openapi_reference OK — 296246 chars loaded.
2026-03-29 22:29:06 [INFO] __main__: Preview: # OpenAPI Specification

## Version 3.2.0

The key words "MUST", "MUST NOT", "REQUIRED", "SHALL", "SHALL NOT", "SHOULD", "SHOULD NOT", "RECOMMENDED", "NOT RECOMMENDED", "MAY", and "OPTIONAL" in this document are to be interpreted as described in [BCP 14](https://tools.ietf.org/html/bcp14) [RFC2119](


In [7]:
# Step 5 — Test reader_node() end-to-end
# Runs the full Reader Node with a real state and inspects all three outputs.

state = {
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
}

result = reader_node(state)

assert "parsed_sections" in result
assert "helper_context" in result
assert "openapi_spec_context" in result
assert isinstance(result["parsed_sections"], list)
assert isinstance(result["helper_context"], str)
assert isinstance(result["openapi_spec_context"], str)

logger.info("reader_node OK")
logger.info(f"  parsed_sections     : {len(result['parsed_sections'])} section(s)")
logger.info(f"  helper_context      : {len(result['helper_context'])} chars")
logger.info(f"  openapi_spec_context: {len(result['openapi_spec_context'])} chars")

if result["parsed_sections"]:
    first = result["parsed_sections"][0]
    logger.info(f"  First section — [{first['section_id']}] {first['title']}")
    logger.info(f"  Content preview   : {first['content'][:200]}")

2026-03-29 22:29:06 [INFO] nodes.reader: Reader Node started.
2026-03-29 22:29:06 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-29 22:29:06 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-29 22:29:06 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-29 22:29:06 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-29 22:29:06 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_spec'.
2026-03-29 22:29:06 [INFO] nodes.reader: Reader Node complete.
2026-03-29 22:29:06 [INFO] __main__: reader_node OK
2026-03-29 22:29:06 [INFO] __main__:   parsed_sections     : 238 section(s)
2026-03-29 22:29:06 [INFO] __main__:   helper_context      : 0 chars
2026-03-29 22:29:06 [INFO] __main__:   openapi_spec